# Практика · Тема 33 · OCR: знайти текст і прочитати

Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
Домашнє завдання: [homework.html](homework.html)

> ⏱ Зошит навчає девʼять маленьких мереж CRNN і ганяє системний рушій `tesseract`.
> Заміряно: **близько чотирьох хвилин** на чотирьох ядрах без відеокарти,
> при `torch.set_num_threads(1)`.

Що зробимо:

1. Оголосимо **власний растровий шрифт 5×7**, щоб усі числа зошита не залежали від
   того, які шрифти стоять у системі.
2. Заміряємо, наскільки погано працює наївний поділ рядка на символи по проміжках.
3. Побудуємо шаблонний читач і подивимось, що з ним робить **неправильна рамка**.
4. Уведемо **відстань Левенштейна** й дві метрики: CER і «рядок цілком».
5. Поставимо промисловий рушій `tesseract` під псування: мова, нахил, розмиття,
   контраст, висота літери, шум.
6. Напишемо крихітну **CRNN** і звіримо власний перебір вирівнювань із бібліотечним
   `nn.CTCLoss`.
7. Знайдемо **вікно за швидкістю навчання**, поза яким CTC віддає порожній рядок.
8. Порівняємо **жадібне декодування** з `beam search`.

In [ ]:
import sys, time, shutil, subprocess
import numpy as np
import torch
import torch.nn as nn

# один потік: на моделях у десятки тисяч ваг багатопотоковість не дає виграшу,
# зате ламає відтворюваність — суми float складаються в іншому порядку
torch.set_num_threads(1)

print("Python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("torch      ", torch.__version__)

# рушій tesseract може бути відсутній на машині читача — тоді зошит
# просто пропустить порівняння, а не впаде
HAVE_TESS = False
TESS_NOTE = ""
try:
    import pytesseract
    version = str(pytesseract.get_tesseract_version()).split()[0]
    langs = pytesseract.get_languages()
    HAVE_TESS = "ukr" in langs
    TESS_NOTE = f"tesseract {version}, мови: {', '.join(sorted(langs))}"
except Exception as err:
    TESS_NOTE = f"tesseract недоступний ({type(err).__name__}) — розділ про рушій буде пропущено"
print("рушій      ", TESS_NOTE)
print("порівняння з tesseract:", "буде" if HAVE_TESS else "пропускаємо")

## 1 · Власний растровий шрифт 5×7

Кожне число цього зошита має відтворитись у тебе. Системні шрифти для цього не
годяться: у мене стоїть Liberation Sans, у тебе може стояти щось інше, і однакового
зображення ми не дістанемо.

Тому весь власний код малює текст **растровим шрифтом, оголошеним прямо тут**:
сітка 5 стовпців на 7 рядків, `#` — чорнило, `.` — папір. Системний шрифт нам
знадобиться рівно один раз — коли ми порівнюватимемо себе з `tesseract`.

In [ ]:
# растровий шрифт 5×7: десять цифр, кожна — сім рядків по пʼять символів
GLYPHS = {
    "0": ["#####", "#...#", "#...#", "#...#", "#...#", "#...#", "#####"],
    "1": ["..#..", ".##..", "..#..", "..#..", "..#..", "..#..", ".###."],
    "2": ["#####", "....#", "....#", "#####", "#....", "#....", "#####"],
    "3": ["#####", "....#", "....#", "#####", "....#", "....#", "#####"],
    "4": ["#...#", "#...#", "#...#", "#####", "....#", "....#", "....#"],
    "5": ["#####", "#....", "#....", "#####", "....#", "....#", "#####"],
    "6": ["#####", "#....", "#....", "#####", "#...#", "#...#", "#####"],
    "7": ["#####", "....#", "....#", "...#.", "..#..", "..#..", "..#.."],
    "8": ["#####", "#...#", "#...#", "#####", "#...#", "#...#", "#####"],
    "9": ["#####", "#...#", "#...#", "#####", "....#", "....#", "#####"],
}
ALPHABET = "0123456789"


def glyph(ch):
    """Одна цифра як масив 7×5 із нулів і одиниць."""
    return np.array([[1.0 if c == "#" else 0.0 for c in row] for row in GLYPHS[ch]],
                    dtype=np.float32)


def draw_line(text, gaps, height=16, top=4, left=2, width=None):
    """Малює рядок цифр. gaps[i] — скільки порожніх стовпців після i-го символу."""
    if width is None:
        width = left * 2 + len(text) * 5 + sum(gaps[:len(text) - 1])
    img = np.zeros((height, width), dtype=np.float32)
    x = left
    for i, ch in enumerate(text):
        if x + 5 > width:
            break
        img[top:top + 7, x:x + 5] = np.maximum(img[top:top + 7, x:x + 5], glyph(ch))
        x += 5 + (gaps[i] if i < len(gaps) else 1)
    return img


def show(img, ink="#", paper="."):
    """Друкує картинку так, як її бачить мережа."""
    for row in img:
        print("".join(ink if v > 0.5 else paper for v in row))


sample = draw_line("4500", [2, 2, 2, 2])
print(f"картинка {sample.shape[0]}×{sample.shape[1]}, чорнила {int(sample.sum())} пікселів")
show(sample)

## 2 · Чому не можна класифікувати літери поодинці

Найочевидніший план читання рядка такий: розріж його на символи, потім класифікуй
кожен символ окремо — задача вже розвʼязана ще в блоці 2.

План ламається на першому ж кроці. Щоб розрізати рядок на символи, треба вже знати,
де вони починаються й закінчуються, — а це та сама задача розпізнавання, тільки
складніша.

Перевіримо це руками. Наївне правило: **порожній стовпець розділяє символи**.
Проженемо його на трьохстах рядках при різних проміжках між цифрами.

In [ ]:
def split_by_empty_columns(img, threshold=0.5):
    """Наївна сегментація: рядок ріжеться там, де стовпець порожній."""
    column_ink = img.max(axis=0)
    pieces, start = [], None
    for x, value in enumerate(column_ink):
        if value > threshold and start is None:
            start = x
        elif value <= threshold and start is not None:
            pieces.append((start, x))
            start = None
    if start is not None:
        pieces.append((start, img.shape[1]))
    return pieces


def gap_experiment(gap_choices, n_lines=300, seed=0):
    """Скільки рядків наївне правило поділило на правильну кількість шматків."""
    rng = np.random.default_rng(seed)
    exact, pieces_total, chars_total = 0, 0, 0
    for _ in range(n_lines):
        length = int(rng.integers(3, 6))
        text = "".join(str(int(d)) for d in rng.integers(0, 10, length))
        gaps = [int(rng.choice(gap_choices)) for _ in range(length)]
        pieces = split_by_empty_columns(draw_line(text, gaps, width=90))
        pieces_total += len(pieces)
        chars_total += length
        exact += (len(pieces) == length)
    return exact / n_lines, pieces_total / chars_total


print("проміжок між цифрами | правильна кількість шматків | шматків на символ")
for choices, name in [((1,), "завжди 1"), ((1, 2), "1 або 2"),
                      ((0, 1, 2), "0, 1 або 2"), ((0, 1), "0 або 1")]:
    ok, ratio = gap_experiment(choices)
    print(f"{name:>20} | {ok:>27.4f} | {ratio:.4f}")

Правило працює **ідеально**, доки між цифрами лишається хоч один порожній стовпець,
і розвалюється, щойно цифри торкаються. Подивімось, які саме пари зливаються.

In [ ]:
merged = []
for first in ALPHABET:
    for second in ALPHABET:
        img = draw_line(first + second, [0, 0], width=20)
        if len(split_by_empty_columns(img)) != 2:
            merged.append(first + second)

print(f"пар цифр, що злилися при нульовому проміжку: {len(merged)} зі 100 "
      f"= {len(merged) / 100:.4f}")
survived = [a + b for a in ALPHABET for b in ALPHABET if a + b not in merged]
print(f"вижили тільки {len(survived)} пар, і в кожній є одиниця: {' '.join(survived)}")
print()
print("«4» і «5» впритул — це один звʼязний шматок:")
show(draw_line("45", [0, 0], width=14))

## 3 · Два кроки: знайти текст і прочитати

OCR — це не одна задача, а дві, поставлені одна за одною:

1. **детекція тексту** — де на знімку рядки;
2. **розпізнавання рядка** — який текст усередині кожної рамки.

Щоб побачити, чому їх розділяють і чим це коштує, збудуємо простий **шаблонний
читач**. Він робить те саме, що будь-який класичний рушій: бере рамку, знаходить у
ній смужку з чорнилом, зводить її до сталої висоти сім пікселів, ріже по порожніх
стовпцях і кожен шматок порівнює з десятьма шаблонами.

In [ ]:
TEMPLATES = {ch: glyph(ch) for ch in ALPHABET}


def nearest_resize(a, height, width):
    """Зміна розміру найближчим сусідом — так рушій зводить смужку до сталої висоти."""
    if a.shape[0] == 0 or a.shape[1] == 0:
        return np.zeros((height, width), dtype=np.float32)
    rows = np.clip((np.arange(height) + 0.5) * a.shape[0] / height, 0, a.shape[0] - 1e-6)
    cols = np.clip((np.arange(width) + 0.5) * a.shape[1] / width, 0, a.shape[1] - 1e-6)
    return a[np.ix_(rows.astype(int), cols.astype(int))]


def read_box(scene, top, bottom, right=None):
    """Прочитати вміст рамки [top, bottom) × [0, right)."""
    box = scene[max(0, top):max(0, bottom), :right]
    if box.shape[0] == 0 or box.max() <= 0.5:
        return ""
    inked = np.where(box.max(axis=1) > 0.5)[0]        # рушій сам шукає рядок у рамці
    band = nearest_resize(box[inked[0]:inked[-1] + 1], 7, box.shape[1])
    out = ""
    for a, b in split_by_empty_columns(band):
        piece = nearest_resize(band[:, a:b], 7, 5)
        out += min(ALPHABET, key=lambda ch: float(np.abs(piece - TEMPLATES[ch]).sum()))
    return out


def build_scene():
    """Знімок оголошення: два рядки цифр — ціна й номер."""
    scene = np.zeros((26, 39), dtype=np.float32)
    for text, top in (("4500", 4), ("12300", 15)):
        x = 3
        for ch in text:
            scene[top:top + 7, x:x + 5] = glyph(ch)
            x += 7
    return scene


SCENE = build_scene()
print("сцена 26×39, перший рядок у рядках 4-10, другий у 15-21")
show(SCENE)
print()
print("рамка точно по рядку [4, 11):", repr(read_box(SCENE, 4, 11)))

In [ ]:
def cer(truth, pred):
    """Посимвольна похибка: відстань Левенштейна, поділена на довжину істини."""
    return levenshtein(truth, pred) / len(truth)


def levenshtein(a, b):
    """Найменша кількість вставок, викидань і замін, щоб перетворити a на b."""
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(previous[j] + 1,          # викинути символ з a
                               current[j - 1] + 1,       # вставити символ з b
                               previous[j - 1] + (ca != cb)))   # лишити або замінити
        previous = current
    return previous[-1]


TRUTH = "4500"
print("як рамка псує читання (істина «4500»)")
print()
print(f"{'рамка':>16} | {'прочитано':>12} | CER")
cases = [(4, 11, None, "точно по рядку"), (2, 13, None, "із запасом 2 px"),
         (0, 15, None, "із запасом 4 px"), (2, 16, None, "зачепила другий рядок"),
         (2, 22, None, "обидва рядки разом"), (6, 11, None, "зрізано верх 2 px"),
         (8, 11, None, "зрізано верх 4 px"), (4, 9, None, "зрізано низ 2 px"),
         (4, 11, 25, "зрізано правий край"), (4, 11, 15, "зрізано половину")]
for top, bottom, right, name in cases:
    pred = read_box(SCENE, top, bottom, right)
    print(f"{name:>22} | {pred:>12} | {cer(TRUTH, pred):.4f}")

Три висновки, і жоден із них не про модель розпізнавання:

- **запас навколо рядка нешкідливий** — читач сам знаходить смужку з чорнилом;
- **зрізаний край рядка коштує рівно стільки символів, скільки зрізано**;
- **дві рядки в одній рамці — повна поразка**, бо смужку зводять до сталої висоти,
  і два рядки, стиснуті в сім пікселів, не схожі ні на що.

Останній випадок і є відповіддю на питання «навіщо детекція окремо»: розпізнавач
уміє читати **один** рядок, і вся його точність тримається на тому, що детектор
дав йому рівно один рядок.

## 4 · Дві метрики, які розходяться

Відстань Левенштейна ми щойно написали. Перевіримо її на прикладах, які можна
порахувати в голові, а потім побудуємо з неї дві метрики.

- **CER** (character error rate, посимвольна похибка) — відстань, поділена на
  довжину істини;
- **рядок цілком** (exact match) — частка рядків, вгаданих без жодної помилки.

In [ ]:
checks = [
    ("телефон", "телефон", 0),       # без помилок
    ("телефон", "телефн", 1),        # викинули одну літеру
    ("телефон", "телефони", 1),      # додали одну
    ("телефон", "тепефон", 1),       # замінили одну: «л» на «п»
    ("телефон", "телефзон", 1),      # вставили зайву «з»
    ("телефон", "тeлeфoн", 3),       # латинські e, e, o замість кирилиць
    ("4500", "45О0", 1),             # латинська O замість нуля
    ("", "абв", 3),
]
for truth, pred, expected in checks:
    got = levenshtein(truth, pred)
    assert got == expected, f"{truth!r} → {pred!r}: {got}, а мало бути {expected}"
    share = got / len(truth) if truth else float("nan")
    print(f"«{truth}» → «{pred}»: відстань {got}, CER {share:.4f}")
print("✅ відстань Левенштейна рахується як треба")

In [ ]:
def corrupt(text, error_rate, rng):
    """Псує кожен символ незалежно з імовірністю error_rate."""
    out = []
    for ch in text:
        if rng.random() >= error_rate:
            out.append(ch)
            continue
        kind = rng.integers(0, 3)
        if kind == 0:
            out.append(ALPHABET[int(rng.integers(0, 10))])   # заміна
        elif kind == 1:
            out.append(ch)
            out.append(ALPHABET[int(rng.integers(0, 10))])   # вставка
        # kind == 2 — викидання, нічого не додаємо
    return "".join(out)


print("одна й та сама модель, різна довжина рядка")
print(f"{'помилка на символ':>18} | {'довжина':>8} | {'CER':>7} | {'рядок цілком':>13}")
for error_rate in (0.02, 0.05):
    for length in (4, 12, 25):
        rng = np.random.default_rng(7)
        truths, preds = [], []
        for _ in range(1000):
            truth = "".join(str(int(d)) for d in rng.integers(0, 10, length))
            truths.append(truth)
            preds.append(corrupt(truth, error_rate, rng))
        score = sum(levenshtein(t, p) for t, p in zip(truths, preds)) / sum(len(t) for t in truths)
        exact = float(np.mean([t == p for t, p in zip(truths, preds)]))
        print(f"{error_rate:>18.2f} | {length:>8} | {score:>7.4f} | {exact:>13.4f}")

CER майже не залежить від довжини рядка, а «рядок цілком» падає з довжиною
експоненційно: та сама модель на чотирисимвольних рядках виглядає добре, а на
двадцятипʼятисимвольних — погано. Обидві метрики чесні, вони просто відповідають на
різні питання, і назвати треба обидві.

## 5 · Промисловий рушій під псуванням

Тепер — єдине місце зошита, де ми беремо системний шрифт і готову навчену модель.
`tesseract` уміє українську, тож можна поставити просте питання: **що йому справді
заважає?**

⚠️ Числа цього розділу залежать від того, який шрифт і яка версія рушія стоять у
системі. На іншій машині вони будуть іншими — важлива не третя цифра, а порядок
величин і порядок причин.

In [ ]:
import os
from PIL import Image, ImageDraw, ImageFont

FONT_CANDIDATES = [
    "/usr/share/fonts/liberation-sans/LiberationSans-Regular.ttf",
    "/usr/share/fonts/liberation/LiberationSans-Regular.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    "/usr/share/fonts/dejavu-sans-fonts/DejaVuSans.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/google-noto/NotoSans-Regular.ttf",
]
SYSTEM_FONT = next((p for p in FONT_CANDIDATES if os.path.exists(p)), None)
RUN_TESS = HAVE_TESS and SYSTEM_FONT is not None

LINES = ["Продам телефон 4500 грн",
         "Ноутбук у гарному стані 12300 грн",
         "Куртка зимова розмір 46 850 грн"]


def render(text, height=32, pad=12):
    """Малює рядок системним шрифтом на білому тлі."""
    font = ImageFont.truetype(SYSTEM_FONT, height)
    probe = ImageDraw.Draw(Image.new("L", (8, 8), 255))
    box = probe.textbbox((0, 0), text, font=font)
    img = Image.new("L", (box[2] - box[0] + 2 * pad, box[3] - box[1] + 2 * pad), 255)
    ImageDraw.Draw(img).text((pad - box[0], pad - box[1]), text, font=font, fill=0)
    return np.array(img)


def ocr(img, lang="ukr", psm=7):
    """psm 7 — «на картинці рівно один рядок тексту»."""
    return pytesseract.image_to_string(img, lang=lang, config=f"--psm {psm}").strip()


def mean_cer(make_image, lang="ukr", psm=7):
    """Середній CER по трьох різних рядках — щоб один щасливий випадок не збрехав."""
    return float(np.mean([cer(line, ocr(make_image(line), lang, psm)) for line in LINES]))


if RUN_TESS:
    print("шрифт:", SYSTEM_FONT)
    print("чистий рядок, українською:")
    for line in LINES:
        got = ocr(render(line))
        print(f"  «{got}»  CER {cer(line, got):.4f}")
else:
    print("рушія або шрифту немає — розділ пропущено")

### Той самий знімок, інший алфавіт

Найкоротший доказ, що **алфавіт — не опція, а частина моделі**: та сама картинка,
той самий рушій, змінено лише мову.

In [ ]:
if RUN_TESS:
    print(f"{'мова':>6} | {'CER рядка 1':>12} | {'середній CER':>13} | що прочиталось")
    for lang in ("ukr", "eng"):
        first = ocr(render(LINES[0]), lang)
        print(f"{lang:>6} | {cer(LINES[0], first):>12.4f} | {mean_cer(render, lang=lang):>13.4f} "
              f"| «{first}»")
else:
    print("пропущено")

Кирилиці в англійському алфавіті немає, тож рушій розкладає кожен гліф на схожі
латинські: **П → N, д → g**, а **ф** узагалі перетворюється на **дві** літери `cb`.
Модель при цьому не «зламалась» — вона чесно віддала найкращу відповідь із тих, що
їй дозволено вимовити.

In [ ]:
import cv2


def rotate(img, degrees):
    """Повертає картинку, розширюючи полотно, щоб нічого не зрізалось."""
    h, w = img.shape
    m = cv2.getRotationMatrix2D((w / 2, h / 2), degrees, 1.0)
    cos, sin = abs(m[0, 0]), abs(m[0, 1])
    nw, nh = int(h * sin + w * cos), int(h * cos + w * sin)
    m[0, 2] += nw / 2 - w / 2
    m[1, 2] += nh / 2 - h / 2
    return cv2.warpAffine(img, m, (nw, nh), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=255)


if RUN_TESS:
    print("нахил рядка")
    print(f"{'градусів':>9} | {'CER':>7} | після вирівнювання назад")
    for deg in (0, 3, 5, 6, 7, 8, 10, 15):
        crooked = mean_cer(lambda line, d=deg: rotate(render(line), d))
        fixed = mean_cer(lambda line, d=deg: rotate(rotate(render(line), d), -d))
        print(f"{deg:>9} | {crooked:>7.4f} | {fixed:.4f}")
else:
    print("пропущено")

In [ ]:
if RUN_TESS:
    print("розмиття при різній висоті літери (CER)")
    print(f"{'висота':>7} |" + "".join(f"{'σ=' + str(s):>9}" for s in (0, 1, 2, 3, 4, 6)))
    for height in (20, 32, 48):
        row = []
        for sigma in (0, 1, 2, 3, 4, 6):
            row.append(mean_cer(lambda line, s=sigma, h=height:
                                render(line, h) if s == 0
                                else cv2.GaussianBlur(render(line, h), (0, 0), s)))
        print(f"{height:>7} |" + "".join(f"{v:>9.4f}" for v in row))
else:
    print("пропущено")

In [ ]:
if RUN_TESS:
    print("контраст: множимо глибину чорного на коефіцієнт")
    print(f"{'коефіцієнт':>11} | {'розмах яскравості':>18} | {'CER':>7}")
    for factor in (1.0, 0.5, 0.25, 0.10, 0.05, 0.02):
        def faded(line, f=factor):
            img = render(line).astype(np.float64)
            return np.clip(255 - (255 - img) * f, 0, 255).astype(np.uint8)
        span = int(faded(LINES[0]).max() - faded(LINES[0]).min())
        print(f"{factor:>11.2f} | {span:>18} | {mean_cer(faded):>7.4f}")
else:
    print("пропущено")

In [ ]:
if RUN_TESS:
    print("висота літери в пікселях")
    print(f"{'висота':>7} | {'CER':>7}")
    for height in (32, 20, 16, 14, 12, 11, 10, 9, 8, 7):
        print(f"{height:>7} | {mean_cer(lambda line, h=height: render(line, h)):>7.4f}")
else:
    print("пропущено")

### Шум: пастка одного заміру

Тут легко збрехати самому собі. Заміряємо не один рівень шуму, а сім, і кожен на
трьох зернах генератора — інакше висновок вийде протилежним до правди.

In [ ]:
if RUN_TESS:
    print(f"{'σ шуму':>7} | {'середній CER':>13} | {'найкраще':>9} | {'найгірше':>9}")
    for sigma in (0.05, 0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60):
        values = []
        for line in LINES:
            clean = render(line).astype(np.float64)
            for seed in range(3):
                rng = np.random.default_rng(seed)
                noisy = np.clip(clean + rng.normal(0, 255 * sigma, clean.shape), 0, 255)
                values.append(cer(line, ocr(noisy.astype(np.uint8))))
        print(f"{sigma:>7.2f} | {np.mean(values):>13.4f} | {min(values):>9.4f} | {max(values):>9.4f}")
else:
    print("пропущено")

In [ ]:
if RUN_TESS:
    # чи допомагає «почистити картинку» перед розпізнаванням
    def blurred(line):
        return cv2.GaussianBlur(render(line), (0, 0), 3)

    def blurred_then_otsu(line):
        _, binary = cv2.threshold(blurred(line), 0, 255,
                                  cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return binary

    print(f"розмитий рядок як є        CER {mean_cer(blurred):.4f}")
    print(f"той самий після Оцу        CER {mean_cer(blurred_then_otsu):.4f}")
else:
    print("пропущено")

In [ ]:
if RUN_TESS:
    def render_two(lines, height=32, pad=12, lead=10):
        """Два рядки в одній картинці — саме те, що дає невдалий детектор."""
        font = ImageFont.truetype(SYSTEM_FONT, height)
        probe = ImageDraw.Draw(Image.new("L", (8, 8), 255))
        boxes = [probe.textbbox((0, 0), t, font=font) for t in lines]
        heights = [b[3] - b[1] for b in boxes]
        width = max(b[2] - b[0] for b in boxes)
        img = Image.new("L", (width + 2 * pad, sum(heights) + lead + 2 * pad), 255)
        draw = ImageDraw.Draw(img)
        y = pad
        for text, box, h in zip(lines, boxes, heights):
            draw.text((pad - box[0], y - box[1]), text, font=font, fill=0)
            y += h + lead
        return np.array(img)

    two = render_two([LINES[0], "Ноутбук 12300 грн"])
    truth_two = LINES[0] + " Ноутбук 12300 грн"
    for psm, name in ((7, "рівно один рядок"), (6, "блок тексту")):
        got = ocr(two, psm=psm).replace("\n", " ").strip()
        print(f"два рядки в рамці, psm {psm} ({name}): CER {cer(truth_two, got):.4f}  «{got}»")

    one = render(LINES[0])
    half = one.shape[0] // 2
    top_half = ocr(one[:half, :])
    bottom_half = ocr(one[half:, :])
    print(f"верхня половина рядка: CER {cer(LINES[0], top_half):.4f}  «{top_half}»")
    print(f"нижня половина рядка:  CER {cer(LINES[0], bottom_half):.4f}  «{bottom_half}»")
else:
    print("пропущено")

## 6 · CRNN: смужка ознак замість окремих літер

Поділ на символи не працює, тому мережа його й не робить. **CRNN** влаштована так:

1. згорткові шари стискають картинку по висоті до одного рядка ознак —
   виходить смужка з `T` колонок;
2. двонапрямлений `GRU` читає смужку зліва направо й справа наліво;
3. лінійний шар у кожній із `T` колонок віддає розподіл над алфавітом **плюс один
   зайвий символ** — порожній.

Ніхто не каже мережі, де саме на картинці кожна цифра. Вирівнювання вона знаходить
сама — і за це відповідає втрата CTC.

In [ ]:
IMG_H, IMG_W = 16, 40


def make_sample(rng, text, noise=0.08):
    """Один рядок цифр із випадковим зсувом і випадковими проміжками."""
    gaps = [int(rng.integers(1, 3)) for _ in text]
    img = draw_line(text, gaps, height=IMG_H,
                    top=4 + int(rng.integers(0, 3)),
                    left=2 + int(rng.integers(0, 3)), width=IMG_W)
    return np.clip(img + rng.normal(0, noise, img.shape), 0, 1).astype(np.float32)


def make_dataset(n, seed, noise=0.08):
    rng = np.random.default_rng(seed)
    images, texts = [], []
    for _ in range(n):
        length = int(rng.integers(3, 6))              # рядки з 3-5 цифр
        text = "".join(str(int(d)) for d in rng.integers(0, 10, length))
        images.append(make_sample(rng, text, noise))
        texts.append(text)
    return np.stack(images), texts


class CRNN(nn.Module):
    def __init__(self, n_classes=11, hidden=48, squeeze_width=False):
        super().__init__()
        # порожній символ дістає індекс 0, цифра d — індекс d+1
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d((2, 2)),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d((2, 2 if squeeze_width else 1)),
        )
        self.rnn = nn.GRU(32 * 4, hidden, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(2 * hidden, n_classes)

    def forward(self, x):
        f = self.cnn(x)                                   # B × 32 × 4 × T
        b, c, h, t = f.shape
        f = f.permute(0, 3, 1, 2).reshape(b, t, c * h)    # B × T × 128
        out, _ = self.rnn(f)
        return self.fc(out)                               # B × T × 11


model_probe = CRNN()
steps = model_probe(torch.zeros(1, 1, IMG_H, IMG_W)).shape[1]
print(f"параметрів у мережі: {sum(p.numel() for p in model_probe.parameters())}")
print(f"колонок на виході T = {steps} при картинці {IMG_H}×{IMG_W}")
print("найдовший рядок L = 5 вимагає щонайменше L плюс кількість повторів колонок;")
print("у найгіршому випадку (усі цифри однакові) це 2·L − 1 = 9 — розділ 9 розбирає це докладно")

## 7 · CTC: порожній символ і склеювання повторів

Мережа віддає `T` колонок, а правильна відповідь має довжину `L`, і `T` майже завжди
більше за `L`. CTC розвʼязує це так:

- до алфавіту додається **порожній символ** (blank) — «тут нічого не закінчилось»;
- шлях завдовжки `T` перетворюють на рядок двома правилами: спершу **склеїти сусідні
  однакові символи**, потім **викинути порожні**;
- імовірність рядка — це **сума** ймовірностей усіх шляхів, що дають цей рядок.

Склеювання повторів потрібне тому, що одна цифра займає кілька колонок смужки й
мережа природно повторює її кілька кроків поспіль. А порожній символ потрібен, щоб
можна було записати **справжній** повтор: «55» вимагає порожнього між двома
пʼятірками, інакше склеювання перетворить його на «5».

Звідси й обмеження **T ≥ 2·L + 1**: у найгіршому випадку між кожними двома символами
треба вставити порожній, плюс по одному на краях.

In [ ]:
def collapse(path):
    """Шлях довжини T → рядок: склеїти повтори, викинути порожні (індекс 0)."""
    out, previous = [], -1
    for k in path:
        if k != previous and k != 0:
            out.append(str(k - 1))
        previous = k
    return "".join(out)


def show_path(path):
    """Шлях у людському вигляді: тире — порожній символ, решта — цифри."""
    return " ".join("–" if k == 0 else str(k - 1) for k in path)


for path, comment in [
    ([0, 6, 6, 0, 0, 2, 0], "порожній між різними цифрами"),
    ([6, 6, 6, 6, 2, 2, 0], "жодного порожнього всередині — те саме"),
    ([0, 6, 0, 6, 0, 0, 0], "порожній розділяє два однакові"),
    ([6, 6, 6, 6, 6, 6, 6], "без порожнього повтор злипається"),
]:
    print(f"{show_path(path):>15}  →  «{collapse(path)}»   {comment}")

In [ ]:
def count_alignments(target, n_steps):
    """Скільки шляхів довжини n_steps склеюються рівно в target — перебором по DP."""
    extended = [0]
    for ch in target:
        extended += [int(ch) + 1, 0]                  # b, c1, b, c2, b, …
    n = len(extended)
    reach = np.zeros(n, dtype=object)
    reach[0] = 1
    if n > 1:
        reach[1] = 1
    for _ in range(1, n_steps):
        nxt = np.zeros(n, dtype=object)
        for s in range(n):
            if reach[s] == 0:
                continue
            nxt[s] += reach[s]                                    # лишитись на місці
            if s + 1 < n:
                nxt[s + 1] += reach[s]                            # крок уперед
            if s + 2 < n and extended[s + 2] != 0 and extended[s] != extended[s + 2]:
                nxt[s + 2] += reach[s]                            # перескочити порожній
        reach = nxt
    return int(reach[n - 1] + (reach[n - 2] if n >= 2 else 0))


header = f"{'рядок':>8} | {'L':>2} | {'повторів':>8} | {'найменше T':>10} |"
print(header + "".join(f"{'T=' + str(t):>10}" for t in (4, 6, 10, 20)))
for target in ("45", "450", "455", "555", "45009"):
    repeats = sum(1 for a, b in zip(target, target[1:]) if a == b)
    smallest = next(t for t in range(1, 30) if count_alignments(target, t) > 0)
    row = [count_alignments(target, t) for t in (4, 6, 10, 20)]
    print(f"{target:>8} | {len(target):>2} | {repeats:>8} | {smallest:>10} |"
          + "".join(f"{v:>10}" for v in row))
print()
print("правило видно прямо з таблиці: найменше T дорівнює L плюс кількість повторів")
print("у найгіршому випадку (усі символи однакові) це 2·L − 1, а не 2·L + 1:")
print("  «55555»: L = 5, повторів 4, найменше T =", 
      next(t for t in range(1, 30) if count_alignments("55555", t) > 0))

Нуль у таблиці означає, що рядок **неможливо** записати: колонок не вистачає. Зверни
увагу на «455» — там потрібен порожній між двома пʼятірками, тож поріг у нього вищий,
ніж у «450» тієї самої довжини.

Тепер найважливіша перевірка зошита: **наша сума по всіх шляхах має збігтися з
бібліотечною втратою CTC**.

In [ ]:
def ctc_prob_by_brute_force(log_probs, target):
    """Імовірність рядка як сума по ВСІХ шляхах — прямим перебором."""
    import itertools
    n_steps, n_classes = log_probs.shape
    total = 0.0
    for path in itertools.product(range(n_classes), repeat=n_steps):
        if collapse(path) == target:
            total += float(np.exp(sum(log_probs[t, k] for t, k in enumerate(path))))
    return total


torch.manual_seed(0)
logits = torch.randn(6, 4)                       # 6 кроків, 4 класи: порожній + «0», «1», «2»
log_probs = torch.log_softmax(logits, dim=-1)
target = "12"

ours = ctc_prob_by_brute_force(log_probs.numpy(), target)
library = float(torch.exp(-nn.CTCLoss(blank=0, reduction="sum")(
    log_probs.unsqueeze(1),
    torch.tensor([int(c) + 1 for c in target]),
    torch.tensor([6]), torch.tensor([len(target)]))))

print(f"наш перебір по 4^6 = {4 ** 6} шляхах : {ours:.10f}")
print(f"nn.CTCLoss                          : {library:.10f}")
assert np.isclose(ours, library, atol=1e-7), "розрахунок розійшовся!"
print("✅ збігається — усередині CTC немає магії, лише сума по шляхах")

In [ ]:
# що буде, якщо колонок не вистачить
short = torch.log_softmax(torch.randn(4, 4), dim=-1)     # T = 4 колонки
loss_fn = nn.CTCLoss(blank=0, reduction="sum", zero_infinity=False)

# «001»: L = 3, один повтор, треба 4 колонки — рівно стільки, скільки є
ok = loss_fn(short.unsqueeze(1), torch.tensor([1, 1, 2]),
             torch.tensor([4]), torch.tensor([3]))
# «0011»: L = 4, два повтори, треба 6 колонок — на дві більше, ніж є
bad = loss_fn(short.unsqueeze(1), torch.tensor([1, 1, 2, 2]),
              torch.tensor([4]), torch.tensor([4]))
print(f"T = 4, рядок «001»  (треба 3 + 1 = 4 колонки): втрата = {ok.item():.4f}")
print(f"T = 4, рядок «0011» (треба 4 + 2 = 6 колонок): втрата = {bad.item()}")
print("нескінченність — це не помилка коду, а чесне «такого шляху не існує»")
print("тому в навчанні стоїть zero_infinity=True: неможливий приклад просто не дає градієнта")

## 8 · Вікно за швидкістю навчання

Тепер навчимо мережу. Три швидкості навчання, три зерна на кожну — різниця, менша за
розкид по зернах, різницею не є.

In [ ]:
def encode_targets(texts):
    """Цілі для CTC: усі рядки складені в один вектор плюс їхні довжини."""
    flat = torch.cat([torch.tensor([int(c) + 1 for c in t]) for t in texts])
    lengths = torch.tensor([len(t) for t in texts])
    return flat, lengths


def greedy_decode(log_probs):
    """Найпростіше декодування: у кожній колонці беремо найімовірніший символ."""
    return [collapse(row) for row in log_probs.argmax(-1).numpy()]


def evaluate(model, images, texts):
    model.eval()
    with torch.no_grad():
        log_probs = model(torch.from_numpy(images).unsqueeze(1)).log_softmax(-1)
    preds = greedy_decode(log_probs)
    exact = float(np.mean([p == t for p, t in zip(preds, texts)]))
    score = sum(levenshtein(t, p) for p, t in zip(preds, texts)) / sum(len(t) for t in texts)
    empty = float(np.mean([p == "" for p in preds]))
    return exact, score, empty, preds


def train_crnn(seed, learning_rate, epochs=60, batch=64, n_train=512):
    torch.manual_seed(seed)
    train_x, train_y = make_dataset(n_train, 100 + seed)
    test_x, test_y = make_dataset(200, 900 + seed)
    model = CRNN()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)
    curve = []
    for epoch in range(epochs):
        model.train()
        order = np.random.default_rng(seed * 7 + epoch).permutation(n_train)
        total, batches = 0.0, 0
        for i in range(0, n_train, batch):
            idx = order[i:i + batch]
            x = torch.from_numpy(train_x[idx]).unsqueeze(1)
            texts = [train_y[j] for j in idx]
            flat, lengths = encode_targets(texts)
            log_probs = model(x).log_softmax(-1).permute(1, 0, 2)   # T × B × C
            input_lengths = torch.full((len(texts),), log_probs.shape[0], dtype=torch.long)
            loss = loss_fn(log_probs, flat, input_lengths, lengths)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item()
            batches += 1
        curve.append(total / batches)
    exact, score, empty, preds = evaluate(model, test_x, test_y)
    return {"exact": exact, "cer": score, "empty": empty,
            "curve": curve, "model": model, "test": (test_x, test_y), "pred": preds}


started = time.time()
runs = {}
for learning_rate in (3e-2, 3e-3, 3e-4):
    for seed in (0, 1, 2):
        runs[(learning_rate, seed)] = train_crnn(seed, learning_rate)
        r = runs[(learning_rate, seed)]
        print(f"lr={learning_rate:<7g} зерно {seed}: рядок цілком {r['exact']:.4f}  "
              f"CER {r['cer']:.4f}  порожніх {r['empty']:.4f}  "
              f"остання втрата {r['curve'][-1]:.4f}", flush=True)
print(f"\nдевʼять навчань зайняли {time.time() - started:.0f} с")

In [ ]:
print(f"{'lr':>8} | {'рядок цілком':>22} | {'CER':>22} | {'порожніх':>9}")
for learning_rate in (3e-2, 3e-3, 3e-4):
    exact = [runs[(learning_rate, s)]["exact"] for s in (0, 1, 2)]
    score = [runs[(learning_rate, s)]["cer"] for s in (0, 1, 2)]
    empty = [runs[(learning_rate, s)]["empty"] for s in (0, 1, 2)]
    print(f"{learning_rate:>8g} | {np.mean(exact):.4f} ±{np.std(exact):.4f} "
          f"[{min(exact):.4f}..{max(exact):.4f}] | "
          f"{np.mean(score):.4f} ±{np.std(score):.4f} [{min(score):.4f}..{max(score):.4f}] | "
          f"{np.mean(empty):>9.4f}")

По обидва боки від робочого вікна — **порожній рядок**. Замала швидкість не дає
мережі вибратись із порожнього символу взагалі (порожніх 1.0000, CER рівно 1.0000);
завелика збиває навчання так, що частина зерен теж падає в порожнечу.

Подивімось на криву втрати — саме там видно, чому це стається.

In [ ]:
titles = ["lr=%g" % lr for lr in (3e-2, 3e-3, 3e-4)]
print(f"{'епоха':>6} |" + "".join(f"{t:>12}" for t in titles))
for epoch in range(0, 60, 5):
    print(f"{epoch:>6} |" + "".join(f"{runs[(lr, 0)]['curve'][epoch]:>12.4f}"
                                    for lr in (3e-2, 3e-3, 3e-4)))
print(f"{59:>6} |" + "".join(f"{runs[(lr, 0)]['curve'][59]:>12.4f}"
                             for lr in (3e-2, 3e-3, 3e-4)))
print()
best = runs[(3e-3, 0)]["curve"]
plateau = [round(v, 4) for v in best[4:14]]
print("робоче навчання, епохи 4-13:", plateau)
print(f"за десять епох втрата впала лише на {best[4] - best[13]:.4f} — це полиця")
print("на цій полиці мережа віддає порожній рядок: він дешевший за будь-яку спробу")
print(f"втрата замалого кроку на 59-й епосі: {runs[(3e-4, 0)]['curve'][59]:.4f} — "
      f"вона з полиці так і не зійшла")

## 9 · Декодування: жадібне проти `beam search`

Жадібне декодування бере в кожній колонці найімовірніший символ. Це швидко, але
неправильно: імовірність **рядка** — це сума по всіх шляхах, а жадібний вибір іде
по одному шляху.

Ось найменший приклад, де вони розходяться. Три класи: порожній, «0», «1». У кожній
колонці порожній трохи ймовірніший за кожну окрему цифру.

In [ ]:
def beam_decode(log_probs, width=8):
    """beam search по префіксах: тримаємо width найкращих рядків."""
    def logaddexp(a, b):
        if a == -np.inf:
            return b
        if b == -np.inf:
            return a
        m = max(a, b)
        return m + np.log(np.exp(a - m) + np.exp(b - m))

    n_steps, n_classes = log_probs.shape
    # для кожного префікса тримаємо дві ймовірності: шлях закінчився порожнім і символом
    beams = {(): (0.0, -np.inf)}
    for t in range(n_steps):
        nxt = {}
        for prefix, (p_blank, p_symbol) in beams.items():
            p_all = logaddexp(p_blank, p_symbol)
            cell = nxt.get(prefix, (-np.inf, -np.inf))
            # додаємо порожній: префікс не змінюється
            nxt[prefix] = (logaddexp(cell[0], p_all + log_probs[t, 0]), cell[1])
            for k in range(1, n_classes):
                p = log_probs[t, k]
                if prefix and prefix[-1] == k - 1:
                    # той самий символ: продовжує попередній, префікс не росте
                    cell = nxt.get(prefix, (-np.inf, -np.inf))
                    nxt[prefix] = (cell[0], logaddexp(cell[1], p_symbol + p))
                    # а другий такий самий символ можливий лише через порожній
                    longer = prefix + (k - 1,)
                    cell = nxt.get(longer, (-np.inf, -np.inf))
                    nxt[longer] = (cell[0], logaddexp(cell[1], p_blank + p))
                else:
                    longer = prefix + (k - 1,)
                    cell = nxt.get(longer, (-np.inf, -np.inf))
                    nxt[longer] = (cell[0], logaddexp(cell[1], p_all + p))
        ranked = sorted(nxt.items(), key=lambda kv: -logaddexp(kv[1][0], kv[1][1]))
        beams = dict(ranked[:width])
    best = max(beams.items(), key=lambda kv: logaddexp(kv[1][0], kv[1][1]))
    return "".join(str(c) for c in best[0])


import itertools

def all_string_probs(log_probs):
    """Повна ймовірність КОЖНОГО рядка — перебором усіх шляхів."""
    n_steps, n_classes = log_probs.shape
    totals = {}
    for path in itertools.product(range(n_classes), repeat=n_steps):
        text = collapse(path)
        weight = float(np.exp(sum(log_probs[t, k] for t, k in enumerate(path))))
        totals[text] = totals.get(text, 0.0) + weight
    return totals


print(f"{'T':>3} | {'p(порожній)':>12} | {'жадібне':>9} | {'його p':>9} | "
      f"{'найкращий':>10} | {'його p':>9} | {'beam':>9}")
for n_steps in (2, 4, 6):
    for p_blank in (0.30, 0.40, 0.50, 0.60):
        rest = 1 - p_blank
        # два символи нерівноймовірні, щоб не було нічиїх
        frame = np.log(np.array([p_blank, rest * 0.6, rest * 0.4]))
        toy = np.tile(frame, (n_steps, 1))
        totals = all_string_probs(toy)
        greedy = collapse(toy.argmax(-1))
        best = max(totals, key=totals.get)
        beam = beam_decode(toy, width=8)
        mark = " " if greedy == best else "  ← розійшлись"
        print(f"{n_steps:>3} | {p_blank:>12.2f} | {('«' + greedy + '»'):>9} | "
              f"{totals.get(greedy, 0.0):>9.5f} | {('«' + best + '»'):>10} | "
              f"{totals[best]:>9.5f} | {('«' + beam + '»'):>9}{mark}")

Тепер те саме на справжній мережі. Візьмемо навчену модель і подамо їй картинки з
дедалі більшим шумом — саме там, де мережа невпевнена, декодування має значення.

In [ ]:
best_model = runs[(3e-3, 0)]["model"]
best_model.eval()
print(f"{'σ шуму':>7} | {'жадібне CER':>12} | {'beam CER':>12} | {'розійшлись':>11}")
for sigma in (0.08, 0.25, 0.40, 0.55):
    images, texts = make_dataset(100, 777, noise=sigma)
    with torch.no_grad():
        log_probs = best_model(torch.from_numpy(images).unsqueeze(1)).log_softmax(-1)
    g = greedy_decode(log_probs)
    b = [beam_decode(log_probs[i].numpy(), width=8) for i in range(len(texts))]
    total = sum(len(t) for t in texts)
    cer_g = sum(levenshtein(t, p) for p, t in zip(g, texts)) / total
    cer_b = sum(levenshtein(t, p) for p, t in zip(b, texts)) / total
    disagree = sum(1 for x, y in zip(g, b) if x != y)
    print(f"{sigma:>7.2f} | {cer_g:>12.4f} | {cer_b:>12.4f} | {disagree:>7}/100")

Результат чесний і не дуже мальовничий: **`beam search` майже нічого не дає**.
Він міняє відповідь на десятках рядків, але CER рухається в третьому знаку, а іноді
й гіршає.

Так і має бути. Промінь виграє не сам по собі, а **разом із мовною моделлю**: коли
до ймовірності шляху додається ймовірність слова, `beam search` може обрати рідший шлях
заради частішого слова. Без словника він просто акуратніше рахує те саме.

## 10 · Що зібрали

Коротко, усе виміряне вище — в одному місці.

In [ ]:
print("ЩО ЛАМАЄ ЧИТАННЯ, за спаданням шкоди")
if RUN_TESS:
    def tiny(line):
        return render(line, 8)

    def crooked(line):
        return rotate(render(line), 15)

    def pale(line):
        img = render(line).astype(np.float64)
        return np.clip(255 - (255 - img) * 0.02, 0, 255).astype(np.uint8)

    wrong_alphabet = mean_cer(render, lang="eng")
    print(f"  чужий алфавіт (eng замість ukr)  CER {wrong_alphabet:.4f}")
    print(f"  висота літери 8 px               CER {mean_cer(tiny):.4f}")
    print(f"  нахил 15 градусів                CER {mean_cer(crooked):.4f}")
    print(f"  контраст 2 % від початкового     CER {mean_cer(pale):.4f}")
else:
    print("  (рушія немає — рядки пропущено)")
print()
ok, ratio = gap_experiment((0, 1, 2))
print(f"НАЇВНИЙ ПОДІЛ ПО ПРОМІЖКАХ: правильних рядків {ok:.4f}")
print(f"РАМКА З ДВОМА РЯДКАМИ:      CER {cer('4500', read_box(SCENE, 2, 22)):.4f}")
print()
for learning_rate in (3e-2, 3e-3, 3e-4):
    exact = [runs[(learning_rate, s)]["exact"] for s in (0, 1, 2)]
    score = [runs[(learning_rate, s)]["cer"] for s in (0, 1, 2)]
    print(f"CRNN lr={learning_rate:<7g} рядок цілком {np.mean(exact):.4f}  CER {np.mean(score):.4f}")

## Завдання

### 🟢 Рівень 1

Додай до растрового шрифту три літери — скажімо, `А`, `В`, `С` — і повтори замір
наївного поділу по проміжках. **Зроблено, якщо** ти назвав частку правильно
поділених рядків для нового алфавіту й пояснив, чому вона змінилась саме так.

### 🟡 Рівень 2

Замір швидкості навчання зроблено при `batch = 64`. Повтори його при `batch = 16` і
`batch = 256`, по три зерна. **Зроблено, якщо** ти показав, що робоче вікно за
швидкістю навчання **зсувається** разом із розміром пакета, і назвав межі для обох.

### 🔴 Рівень 3

Додай до `beam search` найпростішу мовну модель: рядки, у яких є підрядок
`00`, дістають множник 0.5, решта — 1.0. Перевір на шумі 0.40 і 0.55.
**Зроблено, якщо** ти назвав CER до і після й сказав, чи різниця більша за розкид
по трьох зернах.